# 📥 01 — Vehicle Dataset Download
## SmartMine Vision AI · Module 2: Vehicle Detection

---

### Objectives
1. Verify the four Roboflow Universe vehicle datasets are available locally.
2. Download any missing dataset automatically (requires `ROBOFLOW_API_KEY`).
3. Validate the downloaded directory structure.
4. Document sources and license information.

---

### Dataset Sources

| Alias | URL | Format |
|---|---|---|
| `construction_vehicles` | [universe.roboflow.com/0925/construction-vehicle-inspection](https://universe.roboflow.com/0925/construction-vehicle-inspection) | YOLOv8 |
| `mining_area_detection` | [universe.roboflow.com/septiana-s-workspace/mining-area-vehicle-detection](https://universe.roboflow.com/septiana-s-workspace/mining-area-vehicle-detection) | YOLOv8 |
| `riskalert` | [universe.roboflow.com/personal-q02wc/riskalert-mining](https://universe.roboflow.com/personal-q02wc/riskalert-mining) | YOLOv8 |
| `riskalertai` | [universe.roboflow.com/personal-q02wc/riskalertai-mining](https://universe.roboflow.com/personal-q02wc/riskalertai-mining) | YOLOv8 |

> All four are public datasets on Roboflow Universe.
> A **free** Roboflow account with API key is sufficient to download them.
> Get yours at [app.roboflow.com](https://app.roboflow.com/) → Settings → API Keys.

## 1. Setup

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

VEHICLES_DIR = PROJECT_ROOT / "datasets" / "raw" / "vehicles"
print(f"Project root : {PROJECT_ROOT}")
print(f"Vehicles dir : {VEHICLES_DIR}")

In [ ]:
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

SOURCES = {
    "construction_vehicles": {
        "workspace": "0925",
        "project":   "construction-vehicle-inspection",
        "version":   1,
        "url": "https://universe.roboflow.com/0925/construction-vehicle-inspection",
    },
    "mining_area_detection": {
        "workspace": "septiana-s-workspace",
        "project":   "mining-area-vehicle-detection",
        "version":   1,
        "url": "https://universe.roboflow.com/septiana-s-workspace/mining-area-vehicle-detection",
    },
    "riskalert": {
        "workspace": "personal-q02wc",
        "project":   "riskalert-mining",
        "version":   1,
        "url": "https://universe.roboflow.com/personal-q02wc/riskalert-mining",
    },
    "riskalertai": {
        "workspace": "personal-q02wc",
        "project":   "riskalertai-mining",
        "version":   1,
        "url": "https://universe.roboflow.com/personal-q02wc/riskalertai-mining",
    },
}

## 2. Check What Is Already Present

In [ ]:
def images_in(path: Path) -> int:
    return sum(1 for _ in path.rglob("*.jpg")) + sum(1 for _ in path.rglob("*.png"))

print("Dataset availability check")
print("=" * 60)
missing = []
for alias, meta in SOURCES.items():
    dest = VEHICLES_DIR / alias
    n = images_in(dest) if dest.exists() else 0
    status = f"OK  ({n:,} images)" if n > 0 else "MISSING"
    print(f"  {alias:<30} {status}")
    if n == 0:
        missing.append(alias)

print()
if not missing:
    print("All datasets are present. Skip to section 4.")
else:
    print(f"Missing: {missing}")
    print("Run section 3 to download.")

## 3. Download Missing Datasets

In [ ]:
import subprocess, sys

if missing:
    api_key = os.environ.get("ROBOFLOW_API_KEY", "").strip()
    if not api_key or api_key == "your_roboflow_api_key_here":
        print(
            "ROBOFLOW_API_KEY not set.\n"
            "1. Get a free key at https://app.roboflow.com/ → Settings → API Keys\n"
            "2. Add it to .env: ROBOFLOW_API_KEY=<your_key>\n"
            "3. Re-run this cell.\n"
            "\nSkipping download."
        )
    else:
        print("Downloading missing datasets...")
        result = subprocess.run(
            [sys.executable, str(PROJECT_ROOT / "scripts" / "download_datasets.py")],
            capture_output=False,
        )
        if result.returncode != 0:
            print("\n[WARN] Some datasets may not have downloaded correctly.")
            print("Check the output above and verify version numbers in scripts/download_datasets.py.")
else:
    print("Nothing to download.")

## 4. Validate Directory Structure

In [ ]:
print("Post-download validation")
print("=" * 60)

all_ok = True
for alias, meta in SOURCES.items():
    dest = VEHICLES_DIR / alias
    n = images_in(dest) if dest.exists() else 0
    yaml_files = list(dest.rglob("*.yaml")) if dest.exists() else []
    has_yaml = bool(yaml_files)

    ok = n > 0
    all_ok = all_ok and ok
    status = "OK" if ok else "MISSING"
    yaml_status = f"data.yaml found" if has_yaml else "no data.yaml yet"
    print(f"  {alias:<30} [{status}]  {n:,} images  {yaml_status}")
    if ok:
        print(f"    URL: {meta['url']}")

print()
if all_ok:
    print("All four datasets are available. Proceed to 02_dataset_exploration.ipynb.")
else:
    print("Some datasets are still missing. Check your ROBOFLOW_API_KEY and retry.")

## 5. License Summary

| Dataset | License | Attribution |
|---|---|---|
| construction_vehicles | CC BY 4.0 | Roboflow Universe — workspace `0925` |
| mining_area_detection | CC BY 4.0 | Roboflow Universe — Septiana S. |
| riskalert | CC BY 4.0 | Roboflow Universe — personal-q02wc |
| riskalertai | CC BY 4.0 | Roboflow Universe — personal-q02wc |

> Verify the exact license on each project's Roboflow Universe page before
> any commercial use. CC BY 4.0 requires attribution.

## 6. Next Steps

Run `notebooks/02_vehicle_detection/02_dataset_exploration.ipynb` to
explore class distributions and image statistics for the downloaded sources.